In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

import os
import sys
# Add parent directory to path to import modules from one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

# Grid search over k-center matching hyperparameters + OCT training
# Find best configuration based on PR-AUC
import kcenter_hyperparameter_search_global
import importlib
importlib.reload(kcenter_hyperparameter_search_global)
from kcenter_hyperparameter_search_global import run_global_kcenter_matching, build_undersampled_dataset
import model_IAI
importlib.reload(model_IAI)
from model_IAI import finetune_oct_impute, evaluate_binary_oct, get_preprocessor_with_impute
from itertools import product
def load_aps(train_csv_path, test_csv_path=None):
    """
    Loads APS Failure dataset CSV(s).
    Handles:
      - missing value token "na"
      - target column "class" with values pos/neg
      - converts features to numeric
      - adds ENROLID
    Returns: train_df, (optional) test_df
    """
    def _load_one(path):
        df = pd.read_csv(path, na_values=["na", "NA", "NaN", ""], low_memory=False, skiprows=20)
        # Standardize column name
        if "Class" in df.columns and "class" not in df.columns:
            df = df.rename(columns={"Class": "class"})
        assert "class" in df.columns, f"Expected 'class' column, got {df.columns[:5]}..."

        # Map labels
        df["target"] = (df["class"].astype(str).str.lower() == "pos").astype(int)

        # Add ENROLID
        df["ENROLID"] = np.arange(1, len(df) + 1, dtype=np.int64)

        # Drop original label column (keep target)
        df = df.drop(columns=["class"])

        # Convert all remaining (non-ID/non-target) columns to numeric
        feature_cols = [c for c in df.columns if c not in ["ENROLID", "target"]]
        for c in feature_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        return df

    train_df = _load_one(train_csv_path)
    test_df = _load_one(test_csv_path) if test_csv_path is not None else None

    print("\n=== APS LOADED ===")
    print("Train shape:", train_df.shape)
    print("Train target counts:\n", train_df["target"].value_counts())

    if test_df is not None:
        print("Test shape:", test_df.shape)
        print("Test target counts:\n", test_df["target"].value_counts())

    return train_df, test_df
def setup_feature_columns_aps(df, target_col="target"):
    """
    APS: typically all features are numeric after coercion.
    """
    feature_cols = [c for c in df.columns if c not in ["ENROLID", target_col]]

    # After coercion, treat as numeric
    cat_cols = df[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
    if len(cat_cols) > 0:
        # In APS, this usually indicates you missed numeric coercion.
        print("⚠️ Warning: found non-numeric feature cols (expected none):", cat_cols[:10])

    numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

    # Binary detection (rare in APS but keep consistent with your framework)
    bin_cols = []
    for col in numeric_cols:
        u = df[col].dropna().unique()
        if len(u) <= 2 and set(u).issubset({0, 1, 0.0, 1.0}):
            bin_cols.append(col)

    true_num_cols = [c for c in numeric_cols if c not in bin_cols]

    print(f"\n=== APS COLUMN SETUP ===")
    print(f"Total features: {len(feature_cols)}")
    print(f"Categorical columns: {len(cat_cols)}")
    print(f"Binary columns: {len(bin_cols)}")
    print(f"True numeric columns: {len(true_num_cols)}")

    return feature_cols, cat_cols, true_num_cols, bin_cols

def create_train_val_split(df_train, target_col="target", val_size=0.2, random_state=123):
    train_df, val_df = train_test_split(
        df_train,
        test_size=val_size,
        stratify=df_train[target_col],
        random_state=random_state
    )
    print("\n=== APS TRAIN/VAL SPLIT ===")
    print(f"Train: {len(train_df):,} | pos={(train_df[target_col]==1).sum():,}")
    print(f"Val:   {len(val_df):,} | pos={(val_df[target_col]==1).sum():,}")
    return train_df, val_df


from precompute_distances import compute_distances_batched, save_distances_hdf5
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

def get_preprocessor_with_impute(categorical_cols, numeric_cols, verbose=True):
    if verbose:
        print("→ Building preprocessor w/ imputation:")
        print(f"   • Cat: impute(most_frequent) + OHE on: {categorical_cols}")
        print(f"   • Num: impute(median) + scale on: {numeric_cols}")
   
    transformers = []

    if categorical_cols:
        cat_pipe = Pipeline(steps=[
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore")),
        ])
        transformers.append(("cat", cat_pipe, categorical_cols))

    if numeric_cols:
        num_pipe = Pipeline(steps=[
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ])
        transformers.append(("num", num_pipe, numeric_cols))
    
    return ColumnTransformer(transformers=transformers, remainder="drop")


def precompute_case_control_distances(
    train_df, target_col, feature_cols,
    cat_columns, true_num_columns,
    dataset_name, seed=123
):
    import os
    import numpy as np
    from precompute_distances import compute_distances_batched, save_distances_hdf5

    cases = train_df[train_df[target_col] == 1].copy()
    controls = train_df[train_df[target_col] == 0].copy()

    X_cases = cases[feature_cols].copy()
    X_controls = controls[feature_cols].copy()

    print(f"Cases (minority): {len(cases):,}")
    print(f"Controls (majority): {len(controls):,}")

    used_cat = [c for c in cat_columns if c in X_controls.columns]
    used_num = [c for c in true_num_columns if c in X_controls.columns]

    # Ensure we don't drop any features unintentionally
    covered = set(used_cat + used_num)
    dropped = [c for c in feature_cols if c not in covered]
    assert len(dropped) == 0, f"These features would be dropped by the preprocessor: {dropped[:10]}"

    preprocessor = get_preprocessor_with_impute(
        categorical_cols=used_cat,
        numeric_cols=used_num,
        verbose=True
    )

    X_controls_processed = preprocessor.fit_transform(X_controls).astype(np.float32)
    X_cases_processed = preprocessor.transform(X_cases).astype(np.float32)

    assert np.isfinite(X_controls_processed).all()
    assert np.isfinite(X_cases_processed).all()

    distances = compute_distances_batched(
        X_controls_processed, X_cases_processed,
        batch_size=1000, dtype=np.float32
    )

    print(f"  Distance matrix shape: {distances.shape}")
    print(f"  Distance range: [{distances.min():.3f}, {distances.max():.3f}]")
    print(f"  Distance mean: {distances.mean():.3f}")

    output_dir = "./precomputed_distances"
    os.makedirs(output_dir, exist_ok=True)
    h5_path = os.path.join(output_dir, f"distances_{dataset_name}_seed_{seed}.h5")

    majority_enrolids = controls["ENROLID"].to_numpy()
    minority_enrolids = cases["ENROLID"].to_numpy()

    save_distances_hdf5(distances, majority_enrolids, minority_enrolids, h5_path, compression="gzip")
    print(f"\n✓ Saved distances to: {h5_path}")

    return h5_path, X_controls_processed, majority_enrolids


In [2]:
TRAIN_TEST_SEED = 123

aps_train_path = "aps_failure_training_set.csv"
aps_test_path  = "aps_failure_test_set.csv"   # keep the official test set

train_aps_full, test_aps = load_aps(aps_train_path, aps_test_path)
feature_cols_aps, CAT_COLS_APS, TRUE_NUM_COLS_APS, BIN_COLS_APS = setup_feature_columns_aps(train_aps_full)
train_aps, val_aps = create_train_val_split(train_aps_full, val_size=0.2, random_state=TRAIN_TEST_SEED)

X_val = val_aps[feature_cols_aps]
y_val = val_aps["target"]
X_test = test_aps[feature_cols_aps]
y_test = test_aps["target"]


=== APS LOADED ===
Train shape: (60000, 172)
Train target counts:
 target
0    59000
1     1000
Name: count, dtype: int64
Test shape: (16000, 172)
Test target counts:
 target
0    15625
1      375
Name: count, dtype: int64

=== APS COLUMN SETUP ===
Total features: 170
Categorical columns: 0
Binary columns: 0
True numeric columns: 170

=== APS TRAIN/VAL SPLIT ===
Train: 48,000 | pos=800
Val:   12,000 | pos=200


In [7]:
assert set(feature_cols_aps) == set(TRUE_NUM_COLS_APS), "Mismatch: some features not included in preprocessing"
h5_path_aps = precompute_case_control_distances(
    train_aps, "target", feature_cols_aps,
    CAT_COLS_APS, TRUE_NUM_COLS_APS,
    dataset_name="aps_failure", seed=TRAIN_TEST_SEED
)

Cases (minority): 800
Controls (majority): 47,200
→ Building preprocessor w/ imputation:
   • Cat: impute(most_frequent) + OHE on: []
   • Num: impute(median) + scale on: ['aa_000', 'ab_000', 'ac_000', 'ad_000', 'ae_000', 'af_000', 'ag_000', 'ag_001', 'ag_002', 'ag_003', 'ag_004', 'ag_005', 'ag_006', 'ag_007', 'ag_008', 'ag_009', 'ah_000', 'ai_000', 'aj_000', 'ak_000', 'al_000', 'am_0', 'an_000', 'ao_000', 'ap_000', 'aq_000', 'ar_000', 'as_000', 'at_000', 'au_000', 'av_000', 'ax_000', 'ay_000', 'ay_001', 'ay_002', 'ay_003', 'ay_004', 'ay_005', 'ay_006', 'ay_007', 'ay_008', 'ay_009', 'az_000', 'az_001', 'az_002', 'az_003', 'az_004', 'az_005', 'az_006', 'az_007', 'az_008', 'az_009', 'ba_000', 'ba_001', 'ba_002', 'ba_003', 'ba_004', 'ba_005', 'ba_006', 'ba_007', 'ba_008', 'ba_009', 'bb_000', 'bc_000', 'bd_000', 'be_000', 'bf_000', 'bg_000', 'bh_000', 'bi_000', 'bj_000', 'bk_000', 'bl_000', 'bm_000', 'bn_000', 'bo_000', 'bp_000', 'bq_000', 'br_000', 'bs_000', 'bt_000', 'bu_000', 'bv_000', 

Computing distances: 100%|██████████| 48/48 [00:00<00:00, 282.19it/s]


  Distance matrix shape: (47200, 800)
  Distance range: [0.015, 880.486]
  Distance mean: 87.468

Saving to HDF5: ./precomputed_distances/distances_aps_failure_seed_123.h5
  ✓ Saved 134.9 MB

✓ Saved distances to: ./precomputed_distances/distances_aps_failure_seed_123.h5


In [9]:
RESULTS_DIR = "./aps_results_one_run"
h5_path_aps = "./precomputed_distances/distances_aps_failure_seed_123.h5"
os.makedirs(RESULTS_DIR, exist_ok=True)

OCT_DEPTHS = [7, 9]
OCT_MINBUCKETS = [50, 100, 120, 150]
OCT_CPS = [1e-5, 1e-4, 1e-3, 1e-2]


In [ ]:
print("\n" + "="*80)
print("APS — CURATED (k-center matching) + OCT")
print("="*80)

# Minimal, stable config first
matching_ratio = 1          # ~1 negative per positive (you can bump to 2–5 later)
seed_method = "centroid"       # or "random" if you want simplest
case_weighting = None       # try "boundary" later if you have it supported
use_adaptive_pool = True

dnn_dir = f"./precomputed_distances/global_dnn_aps_seed_{TRAIN_TEST_SEED}"
os.makedirs(dnn_dir, exist_ok=True)

matching_result = run_global_kcenter_matching(
    train_pd=train_aps,
    target_col="target",
    feature_cols=feature_cols_aps,
    pn_h5_path=h5_path_aps,
    matching_ratio=matching_ratio,
    case_weighting=case_weighting,
    use_adaptive_pool=use_adaptive_pool,
    seed_method=seed_method,
    CAT_COLUMNS=CAT_COLS_APS,
    TRUE_NUM_COLUMNS=TRUE_NUM_COLS_APS,
    COST_COLUMNS=None,
    dnn_out_dir=dnn_dir,
)

undersampled_train = build_undersampled_dataset(
    train_pd=train_aps,
    matching_result=matching_result,
    target_col="target",
    matching_ratio=matching_ratio,
)

print(f"Curated train size: {len(undersampled_train):,} "
      f"(pos={(undersampled_train['target']==1).sum():,}, neg={(undersampled_train['target']==0).sum():,})")

cur_model, cur_params, cur_grid, cur_preproc, cur_featnames = finetune_oct_impute(
    X_train=undersampled_train[feature_cols_aps],
    y_train=undersampled_train["target"],
    X_val=X_val, y_val=y_val,
    categorical_cols=CAT_COLS_APS,
    numeric_cols=TRUE_NUM_COLS_APS,
    depths=OCT_DEPTHS, minbuckets=OCT_MINBUCKETS, cps=OCT_CPS
)


X_test.shape, undersampled_train.shape
cur_test_metrics = evaluate_binary_oct(
    cur_model, X_test, y_test,
    cur_preproc, cur_featnames,
    results_dir=RESULTS_DIR, save_suffix=f"curated_seed_{TRAIN_TEST_SEED}",
     X_val_df = X_val, y_val = y_val)

print("Curated best params:", cur_params)
print("Curated test metrics:", cur_test_metrics)



APS — CURATED (k-center matching) + OCT

GLOBAL K-CENTER MATCHING CONFIGURATION:
  case_weighting: None
  use_adaptive_pool: True
  seed_method: centroid
  matching_ratio: 1:1

Global Statistics:
  Cases (minority): 800
  Controls (majority): 47,200
  Ratio: 59.00:1

  Preprocessing:
    Features: 170
    Preprocessed shape: (47200, 170)

  Preparing control-control distances (global)...
    ✓ Found existing global d_nn files with matching ENROLIDs:
      d_nn matrix: ./precomputed_distances/global_dnn_aps_seed_123/leaf_global_dnn_matrix.npy
      d_nn enrolids: ./precomputed_distances/global_dnn_aps_seed_123/leaf_global_dnn_enrolids.npy

  K-Center Configuration:
    M (candidate pool size): 23,600 / 47,200 (50.0%)
    Cases to match: 800
    Seed method: centroid
    Adaptive pool: True
    Case weighting: None

  Running two-stage k-center matching (1:1)...
  Seed selection method: 'centroid'
    Centroid seed selected: index 38886 (mean dist to cases: 86.6436)
  Auto-computed tau 

### Centroid seed selected: index 38886 (mean dist to cases: 86.6436) -- Best params: (7, 100, 0.01)
Mean matching cost: 58.6415
    Sampling time: 4.97s

In [20]:
RESULTS_DIR = "./aps_results_one_run"
os.makedirs(RESULTS_DIR, exist_ok=True)

OCT_DEPTHS = [7]
OCT_MINBUCKETS = [150]
OCT_CPS = [1e-5]

print("\n" + "="*80)
print("APS — BASELINE OCT (raw imbalanced train)")
print("="*80)

base_model, base_params, base_grid, base_preproc, base_featnames = finetune_oct_impute(
    X_train=train_aps[feature_cols_aps],
    y_train=train_aps["target"],
    X_val=X_val, y_val=y_val,
    categorical_cols=CAT_COLS_APS,
    numeric_cols=TRUE_NUM_COLS_APS,
    depths=OCT_DEPTHS, minbuckets=OCT_MINBUCKETS, cps=OCT_CPS
)
base_test_metrics = evaluate_binary_oct(
    base_model, X_test, y_test,
    base_preproc, base_featnames, X_val_df=X_val, y_val=y_val,
    results_dir=RESULTS_DIR, save_suffix=f"baseline_seed_{TRAIN_TEST_SEED}",
)

print("Baseline best params:", base_params)
print("Baseline test metrics:", base_test_metrics)



APS — BASELINE OCT (raw imbalanced train)
Finetuning OCT (with imputation) for best PR-AUC
→ Building preprocessor w/ imputation:
   • Cat: impute(most_frequent) + OHE on: []
   • Num: impute(median) + scale on: ['aa_000', 'ab_000', 'ac_000', 'ad_000', 'ae_000', 'af_000', 'ag_000', 'ag_001', 'ag_002', 'ag_003', 'ag_004', 'ag_005', 'ag_006', 'ag_007', 'ag_008', 'ag_009', 'ah_000', 'ai_000', 'aj_000', 'ak_000', 'al_000', 'am_0', 'an_000', 'ao_000', 'ap_000', 'aq_000', 'ar_000', 'as_000', 'at_000', 'au_000', 'av_000', 'ax_000', 'ay_000', 'ay_001', 'ay_002', 'ay_003', 'ay_004', 'ay_005', 'ay_006', 'ay_007', 'ay_008', 'ay_009', 'az_000', 'az_001', 'az_002', 'az_003', 'az_004', 'az_005', 'az_006', 'az_007', 'az_008', 'az_009', 'ba_000', 'ba_001', 'ba_002', 'ba_003', 'ba_004', 'ba_005', 'ba_006', 'ba_007', 'ba_008', 'ba_009', 'bb_000', 'bc_000', 'bd_000', 'be_000', 'bf_000', 'bg_000', 'bh_000', 'bi_000', 'bj_000', 'bk_000', 'bl_000', 'bm_000', 'bn_000', 'bo_000', 'bp_000', 'bq_000', 'br_000'

In [11]:
cur_test_metrics = evaluate_binary_oct(
    cur_model, X_test, y_test,
    cur_preproc, cur_featnames,
    results_dir=RESULTS_DIR, save_suffix=f"curated_seed_{TRAIN_TEST_SEED}",
)

print("Curated best params:", cur_params)
print("Curated test metrics:", cur_test_metrics)


Test dataset for OCT application: 16,000 samples
✓ Predictions completed
Computing optimal thresholds on test set (not recommended for hyperparameter tuning)
✓ Saved OCT predictions to: ./aps_results_one_run/predictions/oct_predictions_curated_seed_123.csv
✓ Saved split table (7 splits) to: ./aps_results_one_run/oct_tree_curated_seed_123_splits.csv
AUC score: 0.961
PR-AUC (Average Precision): 0.612
Best MCC: 0.647 @ threshold=0.696445
Sensitivity (Recall) @MCC*: 0.560
Specificity @MCC*: 0.996
Balanced (G-mean) recall: 0.960
Balanced (G-mean) specificity: 0.903
Number of leaves: 8
Curated best params: (7, 100, 1e-05)
Curated test metrics: {'auc': 0.9613702826666666, 'pr_auc': np.float64(0.6123251145629784), 'best_mcc': 0.6470141366009019, 'best_mcc_threshold': 0.6964453247159387, 'recall_mcc': 0.56, 'precision_mcc': 0.7636363636363637, 'optimal_f1': 0.6461538461050297, 'balanced_recall_gmean': 0.96, 'balanced_specificity_gmean': 0.90272, 'precision_gmean': 0.19148936170212766}


In [ ]:
def pick(d, k):
    v = d.get(k, np.nan) if isinstance(d, dict) else np.nan
    try:
        return float(v)
    except:
        return np.nan

metrics_show = ["auc", "pr_auc", "best_mcc", "balanced_recall_gmean", "balanced_specificity_gmean"]
rows = []
for name, d in [("Baseline", base_test_metrics), ("Curated", cur_test_metrics)]:
    rows.append({"model": name, **{m: pick(d, m) for m in metrics_show}})
cmp = pd.DataFrame(rows)
cmp["delta_cur_minus_base"] = np.nan
display(cmp)
print("\nDelta (Curated - Baseline):")
for m in metrics_show:
    print(f"  {m}: {pick(cur_test_metrics,m) - pick(base_test_metrics,m):+.4f}")
